# 1. نظرة عامة وإعداد البيئة

هذا دفتر خط الأساس المقبول، ذو **درجة Kaggle العامة 0.806 بحسب المستخدم**. الدرجة تاريخية؛ لا يحسبها هذا الدفتر. شغّل الخلايا بالترتيب أو اختر **Restart Kernel ثم Run All**. كل مهمة لها إعدادات وتحميل/تدريب وتنبؤ مستقل أدناه.

من PowerShell في جذر المستودع، مرة واحدة:
```powershell
py -3.12 -m venv .venv
.\.venv\Scripts\python.exe -m pip install -r requirements.txt
```
اختر نواة `.venv/Scripts/python.exe`. ضع الملفات الرسمية الأربعة في `data/raw/`: `train.csv` و`test.csv` و`recordings.csv` و`sample_submission.csv`. المدخلات غير مرفوعة إلى Git. أصول T05 الاثنا عشر والجدول الزمني المجمد موجودة في `models/current_baseline/`؛ لا تحتاج تقديمًا قديمًا أو مجلد Downloads.

**مرشح D-only اختياري ومعطل افتراضيًا**. تساوي درجته العامة المعروضة مع الأساس لا يثبت فائدته على الاختبار المخفي. التجارب المرفوضة لا تعمل هنا. لا يوجد تدريب T05 أو رفع أو commit تلقائي.

In [ ]:
from pathlib import Path
import sys, json, time

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "t05_forecast_config.json").is_file() and (p / "src").is_dir()), None)
if ROOT is None:
    raise RuntimeError("افتح الدفتر داخل مستودع madinah-ai-hackathon")
for folder in (ROOT, ROOT / "src", ROOT / ".vendor"):
    if folder.exists():
        sys.path.insert(0, str(folder))
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits
import baseline_tasks as tasks
import baseline_workflow as workflow
from current_baseline import ASSETS

thread_limit = threadpool_limits(limits=4)
started = time.perf_counter()
protected = workflow.snapshot_protected()
DATA = ROOT / "data/raw"
OUTPUT = ROOT / "outputs/current_baseline_manual"
OUTPUT_NAME = "submission_current_baseline.csv"
VERIFY_ACCEPTED = True  # للمرشح: False + اسم ومجلد منفصلان، بعد التقييم
OUTPUT.mkdir(parents=True, exist_ok=True)
print("ROOT:", ROOT)
print("OUTPUT:", OUTPUT)

# 2. تحميل البيانات والتحقق منها

المسارات مشتقة من جذر المستودع. نفحص نسخ الحزم وبصمات المدخلات والأصول المجمدة، ثم المخطط وتكرار المفاتيح وأهداف التقديم. الخلايا المحجوبة لا تدخل جدول المشاهدات. المصدر: `current_baseline.preflight` و`pipeline.load`.

بعد تعديل مصدر خوارزمية، لا تغير البصمة المقبولة لإخفاء اختلاف: احتفظ بوضع التحقق لإعادة الإنتاج، ثم استخدم وضع مرشح مستقل عند إجراء تعديل مقصود.

In [ ]:
project = workflow.load_project(DATA)
train, test, sample = project["train"], project["test"], project["sample"]
targets, feeds = project["targets"], project["feeds"]
print("train:", train.shape, "test:", test.shape, "submission IDs:", len(sample))
display(targets.groupby(["kind", "day"]).size().rename("targets").reset_index())
display(train.head(3))

# 3. المعالجة المشتركة والإعدادات

`pipeline.table` يحول القراءات المحررة إلى جداول دقيقة × كاميرا لكل تسجيل؛ الغياب يبقى NaN. المرجع الإنتاجي للمزج يأتي من العينة الرسمية. في التقييم يعاد حساب مرجع كل طية دون تاريخها؛ لذلك ليست الأرقام المحلية درجة Kaggle.

`baseline_tasks.py` يحتوي دوال الخوارزميات المشتركة مع التقييم، بينما `baseline_workflow.py` ينسق المدخلات وترتيب IDs وتنسيق CSV القديم فقط. بعض أجزاء المصدر مشتركة: `v2_hidden.fit` يخدم HIDDEN ومرجع HIDDEN-FUTURE؛ `v2_missing.interp_at` يخدم معايرة المانحين وHIDDEN. تعديلها قد يؤثر في أكثر من مهمة.

جدول الصلاة المجمد من [Noor Deen](https://www.noor-deen.com/prayer-times/medina/) بتوقيت Asia/Riyadh. ميزاته مواعيد جدولية، وليست أوقات إقامة أو حركة حشود. T05 يستخدم محاذاته المحسوبة الخاصة.

In [ ]:
CONFIG = tasks.default_config()
schedule = workflow.load_schedule(project)
references = project["references"]

def save_settings():
    """ينقل نفس الإعدادات القابلة للتعديل إلى أداة التقييم المحلي."""
    path = OUTPUT / "task_config.json"
    path.write_text(json.dumps(CONFIG, indent=2), encoding="utf-8")
    return path

CONFIG_PATH = save_settings()
display(pd.DataFrame({"task": list(CONFIG), "source": [
    "baseline_tasks.condition_forecast / forecast_value / im_forecast_value",
    "baseline_tasks.fit_hidden / hidden_value",
    "baseline_tasks.fit_hidden_future / hidden_future_value",
    "baseline_tasks.fit_missing / missing_value"]}))

# 4. FORECAST

نتنبأ بعدد الكاميرا بعد `now` بـ1–15 دقيقة. يسمح فقط بقراءات كل الكاميرات عند `now` أو قبله. المفعّل لـA–H: قالب أسبوعي بمحاذاة الصلاة + مجموعة T05 من ستة أعضاء تشترط على انحرافات الكاميرات وتتعلم ترابطها وتأخرها. لـI–M: مرجع آخر قراءة حديثة، ممزوج بوسيط آخر ثلاث قراءات إذا كانت كلها ضمن خمس دقائق.

**أين تعدّل هذه المهمة؟** `baseline_tasks.condition_forecast` و`forecast_value`، ولـI–M `im_forecast_value` التي تستدعي `im_forecast_fallback_trial.predict`. الخوارزمية الأساسية: `t05_forecast.gp.predict_recording/predict`، والقالب `t05_forecast.pipeline.make_template`. هذه الدوال تصل إلى التقييم أيضًا. `predict_forecast` و`t05_local_forecast.predict` مجرد محولين لبيانات الإنتاج والطية.

وزن النموذج وحدّ حداثة I–M ووزن الوسيط فعّالة عند التنبؤ ولا تحتاج إعادة تدريب. **لا نعرض خطوات التدريب أو ells/rank كأزرار تعديل**: إنها ضمن ملفات covariance المحملة ولن تتغير بتغيير إعداد تدريب نصي. تغيير بنية الملاءمة أو ميزات القالب يتطلب إعادة بناء أصول إنتاج وتقييم مستقلة قبل اعتمادها؛ التحميل يرفض ملفًا ناقصًا ولا يدرب بديلًا تلقائيًا.

In [ ]:
# إعدادات FORECAST الفعالة عند التنبؤ
FORECAST_CONFIG = {"model_weight": 1.0, "im_max_three_age": 5, "im_median_weight": 0.5}
assert 0 <= FORECAST_CONFIG["im_max_three_age"] <= 5  # قاعدة حداثة المرجع لا تتجاوز 5
CONFIG["forecast"] = FORECAST_CONFIG
save_settings()
forecast_targets = workflow.task_targets(project, "forecast")
display(forecast_targets[["id", "unit", "minute_of_day", "now_minute_of_day"]].head(3))
display(workflow.input_example(project, forecast_targets, "forecast"))

In [ ]:
# تحميل الأصول المجمدة وبناء القوالب؛ صفر ملاءمات covariance لـT05
forecast_models = tasks.load_forecast(DATA, ASSETS, project["frozen_config"])
print("Ensemble members:", len(forecast_models["members"]), "; fitted parameter sets: 12")

In [ ]:
# تنبؤ هذه المهمة فقط، مع تنسيق متوافق مع التقديم المقبول
forecast_predictions = workflow.forecast_output(project, forecast_models, FORECAST_CONFIG)
display(forecast_predictions[["id", "people"]].head(3))

**التقييم بعد التعديل:** من جذر المستودع:
```powershell
.\.venv\Scripts\python.exe src/evaluate_baseline_task.py --task forecast --config outputs/current_baseline_manual/task_config.json --output-dir outputs/evaluation/forecast_v1
```
نتائج `by_date.csv` و`by_date_camera.csv` و`verification.json` في مجلد التقييم المختار. يلزم محليًا وجود 12 ملفًا مستقلًا في `outputs/local_score/t05_fit_cache/`؛ ليست أصول الإنتاج بديلًا صحيحًا لها، ولا تأتي مع clone. الأداة تتوقف عند غيابها ولا تعيد تدريب T05. هذا هو اختبار A–H التاريخي ذو عينة ملاءمة 96 خلية وتغطية قالب محدودة، وليس تحققًا مستقلاً جديدًا. لا يوجد I–M في هذا الاختبار؛ انظر قيد تقييم I–M في دليل المساهمة أدناه.

# 5. HIDDEN

نقدّر كاميرا مخفية عند الدقيقة نفسها. لا نستخدم أي قراءة للكاميرا المخفية؛ B/G محجوبتان معًا الجمعة وD السبت. يسمح بسائر القراءات المحررة، ومنها استيفاء محلي قصير للمانحين. المفعّل: مزج المانح المرجعي مع Ridge؛ B يستخدم نسخة dropout بوزن 0.75، وG بوزن 0.50، وD يستخدم Ridge بوزن 0.50. نقص المانحين يعيد تقدير المانح المفرد ثم الوسيط التدريبي.

**أين تعدّل هذه المهمة؟** `baseline_tasks.fit_hidden/hidden_value`؛ الميزات والمعايرة في `v2_hidden._design/fit/predict`. تعديل `alpha` يعيد ملاءمة Ridge عند تشغيل خلية التدريب، ولا يغير T05. تغيير وزن المزج لا يحتاج نموذجًا جديدًا. إعادة تشغيل النواة مهمة بعد تعديل ملفات المصدر.

`d_only=True` خيار منفصل، يستخدم `hidden_reconstruction_trial.fit_new/make_features` لغير الخطي D مع حد أدنى مانحين مرصودين، ويترك باقي المهام على إعداداتها. لا يُفعّل افتراضيًا؛ يتطلب إخراج مرشح باسم مستقل وتعطيل مطابقة الأساس في القسم 8.

In [ ]:
HIDDEN_CONFIG = {"alpha": 3000.0, "blends": {
    "cam_B": ["dropout", 0.75], "cam_G": ["dropout", 0.50], "cam_D": ["ridge", 0.50]},
    "d_only": False}
CONFIG["hidden"] = HIDDEN_CONFIG
save_settings()
hidden_targets = workflow.task_targets(project, "hidden")
display(hidden_targets[["id", "unit", "minute_of_day"]].head(3))
display(workflow.input_example(project, hidden_targets, "hidden"))

In [ ]:
# ملاءمة العلاقات من train فقط؛ لا نضيف مشاهدات test إلى التدريب
hidden_models = tasks.fit_hidden(train, HIDDEN_CONFIG, schedule)
print("Hidden units:", list(hidden_models["linear"]))

In [ ]:
hidden_raw = tasks.predict_hidden(hidden_models, hidden_targets, feeds, references, HIDDEN_CONFIG, schedule)
hidden_predictions = workflow.hidden_output(project, hidden_raw, HIDDEN_CONFIG)
display(hidden_predictions[["id", "people"]].head(3))

**التقييم بعد التعديل:**
```powershell
.\.venv\Scripts\python.exe src/evaluate_baseline_task.py --task hidden --config outputs/current_baseline_manual/task_config.json --output-dir outputs/evaluation/hidden_v1
```
الأداة تستبعد تاريخ التقييم بالكامل، وتستدعي `fit_hidden/hidden_value` نفسيهما، وتختبر عدم تأثير إضافة قيم للكاميرات المحجوبة. الدرجات في `by_date.csv` و`by_date_camera.csv`. لا تحتاج ذاكرة T05 لهذه المهمة.

# 6. HIDDEN-FUTURE

نتنبأ بمستقبل كاميرا مخفية بعد `now` بـ1–15 دقيقة؛ لا تاريخ للكاميرا المحجوبة ولا أي قراءة بعد `now`. المفعّل: نموذج HistGradientBoosting مباشر من تاريخ المانحين، بوزن 0.75 مع المرجع. السبت يضيف ميزات زمنية من جدول الصلاة؛ الجمعة تستخدم النسخة المباشرة العادية. لا نستخدم توقعات المانحين المستقبلية الفعلية.

**أين تعدّل هذه المهمة؟** `baseline_tasks.fit_hidden_future/hidden_future_value`. بناء المدخلات في `v2_experiments.direct_feature`، والتدريب/التنبؤ العاديان في `fit_direct/predict_hf`؛ نسخة السبت في `prayer_experiments.fit_direct_variant/predict_direct_variant`.

`model_params` يصل فعلًا إلى النموذجين عند التدريب؛ تغييره يتطلب إعادة خلية التدريب أو Run All، ولا توجد ملفات estimator محفوظة يجب استبدالها. وزن المزج واختيار نسخة السبت إعدادا تنبؤ؛ تعطيل النسخة ثم إعادة تفعيلها يستلزم إعادة خلية التدريب لبنائها.

In [ ]:
HIDDEN_FUTURE_CONFIG = {"weight": 0.75, "saturday_prayer": True,
    "model_params": {"loss": "absolute_error", "max_iter": 100, "learning_rate": 0.06,
        "max_leaf_nodes": 12, "min_samples_leaf": 40, "random_state": 27}}
CONFIG["hidden-future"] = HIDDEN_FUTURE_CONFIG
save_settings()
hidden_future_targets = workflow.task_targets(project, "hidden-future")
display(hidden_future_targets[["id", "unit", "minute_of_day", "now_minute_of_day"]].head(3))
display(workflow.input_example(project, hidden_future_targets, "hidden-future"))

In [ ]:
hidden_future_models = tasks.fit_hidden_future(train, schedule, HIDDEN_FUTURE_CONFIG)
print("Direct and Saturday prayer estimators fitted from training recordings only.")

In [ ]:
hidden_future_raw = tasks.predict_hidden_future(hidden_future_models, hidden_future_targets,
    feeds, references, HIDDEN_FUTURE_CONFIG, schedule)
hidden_future_predictions = workflow.hidden_future_output(project, hidden_future_raw,
    hidden_future_models, HIDDEN_FUTURE_CONFIG)
display(hidden_future_predictions[["id", "people"]].head(3))

**التقييم بعد التعديل:**
```powershell
.\.venv\Scripts\python.exe src/evaluate_baseline_task.py --task hidden-future --config outputs/current_baseline_manual/task_config.json --output-dir outputs/evaluation/hidden_future_v1
```
الملاءمة تستبعد اليوم كاملًا، ثم تستخدم الدوال المشتركة، مع اختبار استبدال جميع المشاهدات بعد `now`. تقارير التاريخ والكاميرا في مجلد التقييم. المرجع المحلي يعاد بناؤه داخل الطية؛ ليس قيم عينة الإنتاج نفسها.

# 7. MISSING

نملأ انقطاعًا داخل تسجيل لكاميرا مرئية عادةً. هذه المهمة تسمح بالمشاهدات المحررة على جانبي الفجوة، ومنها ما بعد الفجوة، ولا تحمل قيد `now` الخاص بالتنبؤ. لا نستخدم أي خلية محجوبة للهدف أو للمانحين.

المفعّل: استيفاء خطي بين الطرفين مع تصحيح من انحرافات المانحين عن خطوط طرفيهم، معاير على train. يعمل على A/B/C/E/G، وبنصف القوة على F، للفجوات ≥6 دقائق مع مانحين مناسبين. D/H وI–M والحالات غير المدعومة تبقى على مرجع الاستيفاء. تجربتا التنعيم وتعلّم شكل الفجوة مرفوضتان وغير مستخدمتين.

**أين تعدّل هذه المهمة؟** `baseline_tasks.fit_missing/missing_value`، والخوارزمية `v2_missing.fit_calibration/predict/deviations/gap_context`. تعديل المعايرة في المصدر يستلزم إعادة خلية التدريب؛ الأوزان والحد الأدنى للفجوة فعّالان عند التنبؤ ولا يحتاجان أصولًا مجمدة جديدة. لا تفعّل I–M اعتمادًا على تدريب A–H دون تحقق مستقل للنقل.

In [ ]:
MISSING_CONFIG = {"min_gap": 6, "weights": {
    "cam_A": 1.0, "cam_B": 1.0, "cam_C": 1.0, "cam_D": 0.0,
    "cam_E": 1.0, "cam_F": 0.5, "cam_G": 1.0, "cam_H": 0.0}}
CONFIG["missing"] = MISSING_CONFIG
save_settings()
missing_targets = workflow.task_targets(project, "missing")
display(missing_targets[["id", "unit", "minute_of_day"]].head(3))
display(workflow.input_example(project, missing_targets, "missing"))

In [ ]:
missing_calibration = tasks.fit_missing(train, MISSING_CONFIG)
print("Calibrated camera pairs:", len(missing_calibration))

In [ ]:
missing_raw = tasks.predict_missing(missing_calibration, missing_targets, feeds, references, MISSING_CONFIG)
missing_predictions = workflow.missing_output(project, missing_raw)
display(missing_raw[["id", "people", "used", "donors", "gap"]].head(3))

**التقييم بعد التعديل:**
```powershell
.\.venv\Scripts\python.exe src/evaluate_baseline_task.py --task missing --config outputs/current_baseline_manual/task_config.json --output-dir outputs/evaluation/missing_v1
```
نفس `fit_missing/missing_value`، باستبعاد تاريخ الطية بالكامل، على الحالات الـ358 الثابتة. توجد 38 فترة انقطاع: 32 كاملة الإجابات و6 جزئية بسبب فقد أصلي؛ لا نحول الدقائق المتجاورة إلى تجارب مستقلة. التقارير في مجلد التقييم، ولا يعيد الأمر تدريب T05.

# 8. جمع التوقعات والتحقق وحفظ التقديم

نجمع نتائج المهام حسب IDs العينة، ثم نفحص التغطية والتفرّد والقيم المنتهية غير السالبة. `baseline_workflow.save_combined` يحافظ على التنسيق التاريخي، ويفحص SHA-256 المقبولة. إن توفر التقديم المحمي محليًا نقارن كل قيمة أيضًا. اختلاف البصمة يوقف وضع إعادة الإنتاج، ولا يعد نجاحًا تلقائيًا.

عند تعديل مقصود، غيّر في القسم 1: `VERIFY_ACCEPTED=False`، و`OUTPUT_NAME` إلى اسم مرشح منفصل مثل `submission_missing_manual_v1.csv`، ويفضل مجلد OUTPUT مستقل. لا تغيّر البصمة المقبولة. لا يُستبدل ملف مختلف موجود مسبقًا.

In [ ]:
CONFIG_PATH = save_settings()
task_predictions = {"forecast": forecast_predictions, "hidden": hidden_predictions,
    "hidden-future": hidden_future_predictions, "missing": missing_predictions}
verification = workflow.save_combined(project, task_predictions, CONFIG, OUTPUT,
    verify_accepted=VERIFY_ACCEPTED, filename=OUTPUT_NAME, protected=protected, started=started)
assert verification["ids"] == 4457 and verification["valid_nonnegative"]
if VERIFY_ACCEPTED:
    assert verification["sha256_matches_accepted"]
display(pd.DataFrame(verification.get("by_task", {})).T)
print(json.dumps(verification, indent=2))
print("Settings for local evaluation:", CONFIG_PATH)

### دليل مساهمة قصير دون Codex

1. افتح PowerShell في جذر المستودع وأنشئ فرعًا: `git switch -c task/missing-update`.
2. عدّل مهمة واحدة في الدالة المذكورة في قسمها، أو عدّل إعداداتها في خلايا الدفتر. لا تعدّل ملفات `models/current_baseline/` عشوائيًا. اختر اسم إخراج مرشح مستقل ووضع `VERIFY_ACCEPTED=False` للتغيير المقصود.
3. **Restart Kernel ثم Run All** لتجنب الاحتفاظ بنماذج أو imports قديمة. الخلايا تعيد تدريب النماذج الصغيرة فقط وتحمّل T05.
4. شغّل أمر تقييم المهمة أعلاه مع **نفس ملف `task_config.json` الذي حفظه تشغيلك**؛ عدّل مساره إذا غيرت OUTPUT. استخدم اسم مجلد تقييم جديد كل مرة. افحص `by_date.csv` و`by_date_camera.csv` و`scored.csv`؛ تحقق من تاريخين وعدة فجوات/نوافذ، لا من مثال واحد.
5. احفظ التقديم باسم مستقل. ثم: `git diff`، و`git add notebooks/current_baseline.ipynb src/<changed_file>.py`، و`git commit -m "Describe the task change"`. مجلدات البيانات والنواتج مستبعدة من Git؛ دوّن مسار المرشح ونتائجه في وصف المساهمة. لا يوجد commit/push/رفع تلقائي.

**الربط بالتقييم:** المصدر والإعدادات المشتركة تصل للدوال نفسها في الدفتر و`evaluate_baseline_task.py`؛ كما يستخدم `local_score.fit_fold/predict_case` دوال المهام نفسها بالإعدادات الافتراضية. الأمر التاريخي `scripts/run_local_score.ps1` يكتب تقاريره القديمة وقد يلائم T05 إذا لم يجد ذاكرته، لذلك استخدم المقيم الجديد للتعديل اليدوي؛ لا تستبدل أصول الإنتاج بذاكرة تقييم.

**قيود واضحة:** اختبار A–H التاريخي يعيد بناء مرجع الطية ويستخدم إعداد ملاءمة T05 صغيرًا مختلفًا عن الإنتاج. ذاكرة forecast غير مضمنة في clone؛ تغييرات ملاءمة T05 تحتاج بناء ذاكرة وأصول جديدة منفصلة قبل ادعاء تقييمها، والمقيم الجديد يرفض التدريب تلقائيًا. لتعديل I–M استخدم المقيم نفسه مع `--task forecast-im` ونفس `--config` في مجلد إخراج جديد؛ يستدعي دالة I–M نفسها دون تدريب. الـ358/3719 حالة التاريخية لا تقيس I–M؛ البروكسي بدوره حالات متداخلة ولا يكشف إجابات الأهداف الحقيقية. التفاصيل في `docs/teammate_baseline.md`. تواريخ التقييم سبق فحصها كثيرًا وليست تحققًا مستقلًا جديدًا. أمثلة المدخلات في الأقسام مقتطفات صغيرة من القراءات المسموحة، وليست عرضًا لكل ميزات النموذج.